# Bakta CTS Demo

End-to-end demo of the cdm_bakta CTS tool.

- **Image:** `ghcr.io/kbaseincubator/cdm_bakta:0.1.3@sha256:ab3d636d9381fdd57a8b21e3a2403663987f57f123027fa7a773474e66d9a827`
- **Refdata UUID:** `30f8ba11-a456-408c-a9f9-7d232ba3ed8e`
- **Refdata file:** `cts-refdata/bakta/v6.0_amr20260324/bakta_db.tar.gz` (~30GB compressed, ~84GB unpacked)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/bakta/test/v3_diamond_v220`

Bakta does full bacterial genome annotation. Input is **nucleotide FASTA** (genome assemblies, `.fna` or `.fna.gz`), unlike kofamscan which needs protein sequences.

The 0.1.3 image overlays diamond v2.2.0 over the conda-shipped diamond v2.1.21 to fix an intermittent deadlock at the pseudogene-detection alignment step. The `BAKTA_DB` env var is set in the image to `/ref_data/db`, so callers do NOT need to pass `--db` on the command line.

Re-executed 2026-11-06 after the tool audit: the TSV parse now reads only the main annotation table (the `.inference.tsv` and `.hypotheticals.tsv` files have different columns), and the checks compare CDS counts per genome with bakta's own summary.

## 1. Setup

In [1]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_bakta:0.1.3@sha256:ab3d636d9381fdd57a8b21e3a2403663987f57f123027fa7a773474e66d9a827"
OUTPUT_DIR = "cts/io/jplfaria/output/bakta/demo_20261106"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': ['cts_user'], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Use the same 4 test genomes the rest of the team has been using (`cts/io/gavin/test_files/`). These are nucleotide assemblies with CRC64NVME checksums.

In [2]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True):
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")

4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. Submit Bakta job

Bakta is heavier than kofamscan: full DB (~84GB unpacked) + lots of subprocess steps (CDS prediction, tRNA/rRNA scans, BLAST against UniRef, etc). Plan for ~20-40 min per bacterial genome at 4 CPUs.

One container per input genome (parallelizes the wall time).

Note on genetic code: bakta defaults to translation table 11. One of the test genomes, GCA_000147015.1 (*Candidatus* Zinderia insecticola), uses code 4 (UGA = Trp), so with the default many of its genes are cut at internal UGA codons. For real use submit code-4 organisms separately with `--translation-table 4`; this demo keeps one job for simplicity.

In [3]:
if not input_files:
    raise RuntimeError("No .fna(.gz) inputs found. Adjust the prefix above.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "--output", "/out",
        "--threads", "4",
        "--keep-contig-headers",
        "--skip-plot",        # skip the circular-genome PNG/SVG (slow + not needed for this test)
        "--force",            # bakta refuses to write to existing /out (CTS pre-creates it)
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="32GB",
    runtime="PT4H",
)
print("Job ID:", job.id)

Job ID: 1f0102bf-5ab4-4da4-b1de-881a118a6075


## 4. Wait for completion


In [4]:
# Block until the job finishes, printing the final status.
import json
result = job.wait_for_completion()
print(json.dumps(result, indent=2, default=str))

{
  "id": "1f0102bf-5ab4-4da4-b1de-881a118a6075",
  "state": "complete",
  "transition_times": [
    {
      "state": "created",
      "time": "2026-11-06T21:17:01.872000Z"
    },
    {
      "state": "download_submitted",
      "time": "2026-11-06T21:17:01.949000Z"
    },
    {
      "state": "job_submitting",
      "time": "2026-11-06T21:17:15.347000Z"
    },
    {
      "state": "job_submitted",
      "time": "2026-11-06T21:17:39.436000Z"
    },
    {
      "state": "upload_submitting",
      "time": "2026-11-06T21:28:34.227000Z"
    },
    {
      "state": "upload_submitted",
      "time": "2026-11-06T21:28:34.506000Z"
    },
    {
      "state": "complete",
      "time": "2026-11-06T21:28:35.124000Z"
    }
  ],
  "user": "jplfaria",
  "admin_meta": {
    "cse_event_processing_start": "2026-11-06T21:28:41.959979+00:00",
    "cse_event_processing_no_operation": "2026-11-06T21:28:42.339723+00:00"
  }
}


## 5. Inspect output files

In [5]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} total output files")
for o in outs:
    print(f"  {o['file']}")

52 total output files
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.log
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.tsv
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.gff3
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.gbff
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.embl
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.fna
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.ffn
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.faa
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.inference.tsv
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.hypotheticals.tsv
  cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1

## 6. Read annotation TSVs as a dataframe

Each genome gets one main table `<prefix>.tsv` with columns
`Sequence Id, Type, Start, Stop, Strand, Locus Tag, Gene, Product, DbXrefs`.
`<prefix>.inference.tsv` and `<prefix>.hypotheticals.tsv` also end in `.tsv` but have different columns, so they are excluded.
Lines starting with `#` are metadata; the table header is the `#Sequence Id` line.

In [6]:
import io
import pandas as pd

tsvs = [o for o in outs if o["file"].endswith(".tsv")
        and not o["file"].endswith((".inference.tsv", ".hypotheticals.tsv"))]
print(f"{len(tsvs)} bakta .tsv annotation file(s)")

frames = []
for o in tsvs:
    bucket, key = o["file"].split("/", 1)
    obj = mincli.get_object(bucket, key)
    raw = obj.read().decode("utf-8")
    # Strip metadata comment lines, find the column header line, then parse the rest
    lines = raw.splitlines()
    header_idx = next((i for i, ln in enumerate(lines) if ln.startswith("#Sequence Id")), None)
    if header_idx is None:
        print(f"  WARN: no header line in {o['file']}")
        continue
    table = "\n".join([lines[header_idx][1:]] + lines[header_idx + 1:])
    df = pd.read_csv(io.StringIO(table), sep="\t")
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal feature rows: {len(all_df)}")
if len(all_df):
    print("\nFeature counts by Type:")
    print(all_df["Type"].value_counts())
all_df.head(20)

4 bakta .tsv annotation file(s)



Total feature rows: 7331

Feature counts by Type:
Type
cds              5849
crispr-spacer     612
crispr-repeat     612
tRNA              165
ncRNA-region       54
crispr             18
rRNA               14
ncRNA               6
tmRNA               1
Name: count, dtype: int64


,Sequence Id,Type,Start,Stop,Strand,Locus Tag,Gene,Product,DbXrefs,source_file
0,AE017199.1,cds,1,879,-,AHLLAC_0001,NaN,(3' truncated) hypothetical protein,PFAM:PF01881.21,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
1,AE017199.1,cds,883,2691,+,AHLLAC_0002,NaN,hypothetical protein,"PFAM:PF00270.35, PFAM:PF00271.37, PFAM:PF19306...",cts/io/jplfaria/output/bakta/demo_20261106/0/G...
2,AE017199.1,cds,2668,3189,-,AHLLAC_0003,NaN,hypothetical protein,PFAM:PF22199.2,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
3,AE017199.1,cds,3290,3748,-,AHLLAC_0004,NaN,hypothetical protein,PFAM:PF01951.21,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
4,AE017199.1,cds,3745,4665,-,AHLLAC_0005,NaN,hypothetical protein,NaN,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
5,AE017199.1,cds,4680,5090,+,AHLLAC_0006,NaN,hypothetical protein,PFAM:PF01035.26,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
6,AE017199.1,cds,5028,5561,-,AHLLAC_0007,NaN,hypothetical protein,NaN,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
7,AE017199.1,cds,5537,5707,-,AHLLAC_0008,NaN,hypothetical protein,NaN,cts/io/jplfaria/output/bakta/demo_20261106/0/G...
8,AE017199.1,cds,5759,7000,+,AHLLAC_0009,NaN,hypothetical protein,"PFAM:PF01938.26, PFAM:PF04055.27",cts/io/jplfaria/output/bakta/demo_20261106/0/G...
9,AE017199.1,tRNA,7001,7074,-,AHLLAC_0010,trnI,tRNA-Ile(gat),SO:0000263,cts/io/jplfaria/output/bakta/demo_20261106/0/G...


## 7. End-to-end check

Per genome, the number of CDS rows in the main table must equal the `CDSs:` count in bakta's `.txt` summary.

In [7]:
import re
tsv_cds = all_df[all_df["Type"] == "cds"].groupby("source_file").size()
summary = {}
for o in outs:
    if o["file"].endswith(".txt"):
        txt = mincli.get_object(*o["file"].split("/", 1)).read().decode("utf-8")
        m = re.search(r"^CDSs:\s*(\d+)", txt, re.M)
        summary[o["file"][:-4] + ".tsv"] = int(m.group(1)) if m else None
cmp = pd.DataFrame({"tsv_cds": tsv_cds, "summary_cds": pd.Series(summary)})
print(cmp.to_string())

checks = {
    "job complete, all exit 0": job.get_job_status()["state"] == "complete"
                                and set(job.get_exit_codes().get("exit_codes", [])) == {0},
    "one main .tsv per input genome": len(tsvs) == len(input_files),
    "every genome has CDS": len(tsv_cds) == len(input_files) and (tsv_cds > 0).all(),
    "CDS rows match bakta summary": len(cmp) == len(input_files) and (cmp["tsv_cds"] == cmp["summary_cds"]).all(),
    "has Product assignments": all_df["Product"].notna().any() if len(all_df) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off, inspect outputs above.")


                                                                                         tsv_cds  summary_cds
cts/io/jplfaria/output/bakta/demo_20261106/0/GCA_000008085.1_ASM808v1_genomic.fna.tsv        569          569
cts/io/jplfaria/output/bakta/demo_20261106/1/GCA_000010565.1_ASM1056v1_genomic.fna.tsv      2981         2981
cts/io/jplfaria/output/bakta/demo_20261106/2/GCA_000145985.1_ASM14598v1_genomic.fna.tsv     1992         1992
cts/io/jplfaria/output/bakta/demo_20261106/3/GCA_000147015.1_ASM14701v1_genomic.fna.tsv      307          307


  [x] job complete, all exit 0
  [x] one main .tsv per input genome
  [x] every genome has CDS
  [x] CDS rows match bakta summary
  [x] has Product assignments

All green.


## 8. Diagnostic (only useful if the job errored)

CTS uploads container logs only for failed jobs, so `print_logs` returns 404 for a successful run.

In [8]:
# Error diagnostic - only run if state is error
ed = job.get_job()
print("logpath:", ed.get("logpath"))
print("error:", ed.get("error"))
print("exit codes:", job.get_exit_codes())
print("---stderr container 0---")
try:
    job.print_logs(container_num=0, stderr=True)
except Exception as e:
    print("stderr unavailable:", e)
print("---stdout container 0---")
try:
    job.print_logs(container_num=0, stderr=False)
except Exception as e:
    print("stdout unavailable:", e)


logpath: None
error: None


exit codes: {'exit_codes': [0, 0, 0, 0]}
---stderr container 0---


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-11-06T21:35:44.429858+00:00', 'request_id': '04f4ad81-3158-47c0-90e7-ad7a279ea80a', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID 1f0102bf-5ab4-4da4-b1de-881a118a6075 has no logs available'}}


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-11-06T21:35:44.495920+00:00', 'request_id': 'c7e53d74-f928-4436-9808-e1444b90d812', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID 1f0102bf-5ab4-4da4-b1de-881a118a6075 has no logs available'}}


stderr unavailable: Job ID 1f0102bf-5ab4-4da4-b1de-881a118a6075 has no logs available
---stdout container 0---
stdout unavailable: Job ID 1f0102bf-5ab4-4da4-b1de-881a118a6075 has no logs available
